# 06 — Indexar protocolos no Chroma

Constroi o vector store dos protocolos clínicos (RAG) a partir do `fontes_saude_mulher_v2.json` gerado em `01_extrair_protocolos.ipynb`.

- **Embeddings**: `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2` (rápido, 384d, suporte a PT-BR)
- **Store**: Chroma persistente em `/content/drive/MyDrive/AssistenteHospitalar/files/chroma/`
- **Chunking**: mesmo do `02_gerar_dataset_sft.ipynb` (6000 char, overlap 400)
- **Metadados**: `doc_id`, `chunk_id`, `category`, `sensitive`, `name`

Demora estimada: ~5 min para 1392 chunks no Colab com GPU.

In [ ]:
!pip install -q chromadb langchain langchain-community langchain-huggingface sentence-transformers

In [ ]:
import os, sys, json, re
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

DRIVE_BASE  = '/content/drive/MyDrive/AssistenteHospitalar'
SOURCE_JSON = f'{DRIVE_BASE}/files/fontes_saude_mulher_v2.json'
CHROMA_DIR  = f'{DRIVE_BASE}/files/chroma'
COLLECTION  = 'protocolos_saude_mulher'

EMB_MODEL   = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
CHUNK_SIZE, CHUNK_OVERLAP = 6000, 400

Path(CHROMA_DIR).mkdir(parents=True, exist_ok=True)
print('Source:', SOURCE_JSON)
print('Chroma:', CHROMA_DIR)

In [ ]:
with open(SOURCE_JSON, 'r', encoding='utf-8') as f:
    docs = json.load(f)
print(f'Documentos: {len(docs)}')

def chunk_text(text, size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    text = re.sub(r'\n{3,}', '\n\n', text).strip()
    if len(text) <= size:
        return [text]
    chunks, start = [], 0
    while start < len(text):
        end = min(start + size, len(text))
        if end < len(text):
            cut = text.rfind('\n\n', start, end)
            if cut > start + size // 2:
                end = cut
        chunks.append(text[start:end].strip())
        if end >= len(text):
            break
        start = end - overlap
    return [c for c in chunks if len(c) > 200]

chunked = []
for d in docs:
    for i, c in enumerate(chunk_text(d['content'])):
        chunked.append({
            'text': c,
            'metadata': {
                'doc_id':    d['filename'],
                'chunk_id':  f"{d['filename']}::{i}",
                'category':  d['category'],
                'sensitive': d['sensitive'],
                'name':      d['name'],
            },
        })
print(f'Chunks: {len(chunked)}')

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document

embeddings = HuggingFaceEmbeddings(
    model_name=EMB_MODEL,
    model_kwargs={'device': 'cuda'},        # use 'cpu' se sem GPU
    encode_kwargs={'normalize_embeddings': True},
)

lc_docs = [Document(page_content=c['text'], metadata=c['metadata']) for c in chunked]

vectorstore = Chroma.from_documents(
    documents=lc_docs,
    embedding=embeddings,
    collection_name=COLLECTION,
    persist_directory=CHROMA_DIR,
)
print(f'Indexados {vectorstore._collection.count()} chunks em {CHROMA_DIR}')

In [ ]:
# Smoke test do retriever
retriever = vectorstore.as_retriever(search_kwargs={'k': 4})
queries = [
    'Quais critérios para repetir citologia em paciente com LSIL?',
    'Profilaxia pós-exposição em violência sexual',
    'Indicações de internação psiquiátrica involuntária',
    'Posologia de misoprostol para aborto retido',
    'Sinais de alerta para violência doméstica',
]
for q in queries:
    print(f'\n>>> {q}')
    for d in retriever.invoke(q):
        m = d.metadata
        print(f'  [{m.get("category")}] {m.get("doc_id")} :: chunk {m.get("chunk_id", "").split("::")[-1]}')
        print(f'    "{d.page_content[:140]}..."')

In [ ]:
# Verifica que o store carrega corretamente do disco (sem reindexar)
from langchain_community.vectorstores import Chroma
vs2 = Chroma(collection_name=COLLECTION, embedding_function=embeddings, persist_directory=CHROMA_DIR)
print('Total indexado (reload):', vs2._collection.count())